# **Project 2: Build the vector store**

New ideas in this notebook (they were not in the LangGraph course):

- **Embedding:** a list of numbers that represents the *meaning* of a text. Texts with similar meaning get similar numbers.
- **Vector store (Chroma):** a database that keeps the embeddings and finds the closest ones to a question.
- **Similarity search:** embed the question, then return the `k` closest chunks.

**Chunk = one section.** Only sections longer than 3000 characters are split, at subsection or sentence boundaries.
Each chunk keeps its section number, title, source and version date as **metadata**, so every answer can cite its source.

In [1]:
import json

with open("../data/sections.json", encoding="utf-8") as f:
    sections = json.load(f)

print(len(sections), "sections loaded")
print(sections[11]["no"], sections[11]["title"])

130 sections loaded
8 Contracts of service not to restrict rights of employees to join, participate in or organize trade unions


### **Step 1: Turn sections into chunks (Documents)**

In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

SOURCE = "Employment Act 1955 (Act 265), Reprint as at 1 August 2023"
VERSION_DATE = "2023-08-01"
MAX_CHARS = 3000

# try to split at a new line (subsection) first, then a sentence end, then ";", and a space only as a last resort
splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=150,
                                          separators=["\n", ". ", "; ", " "])

documents = []

for s in sections:
    header = f"Section {s['no']}. {s['title']}"
    parts = splitter.split_text(s["text"]) if len(s["text"]) > MAX_CHARS else [s["text"]]

    for i, part in enumerate(parts):
        tag = f" (part {i + 1}/{len(parts)})" if len(parts) > 1 else ""
        documents.append(Document(
            page_content=f"{header}{tag}\n{part}",       # the header is repeated in EVERY part
            metadata={"section_no": s["no"], "section_title": s["title"],
                      "source": SOURCE, "version_date": VERSION_DATE, "part": i},
        ))

print(len(sections), "sections ->", len(documents), "chunks")
print(documents[0].page_content[:300])
print(documents[0].metadata)

130 sections -> 168 chunks
Section 1. Short title and application
(1) This Act may be cited as the Employment Act 1955.
(2) This Act shall apply to *Peninsular Malaysia only.
{'section_no': '1', 'section_title': 'Short title and application', 'source': 'Employment Act 1955 (Act 265), Reprint as at 1 August 2023', 'version_date': '2023-08-01', 'part': 0}


### **Step 2: Create the embeddings and the Chroma store**
The embedding model runs on your computer (the first run downloads it). `normalize_embeddings` makes the distance scores easier to compare.

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5", encode_kwargs={"normalize_embeddings": True})

vectorstore = Chroma(
    collection_name="employment_act",
    embedding_function=embeddings,
    persist_directory="../chroma_db",
)

# start fresh so running this notebook twice does not create duplicates
vectorstore.delete_collection()

vectorstore = Chroma(
    collection_name="employment_act",
    embedding_function=embeddings,
    persist_directory="../chroma_db",
)

vectorstore.add_documents(documents)
print(vectorstore._collection.count(), "chunks stored")

c:\Users\User\employ-law-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4363.24it/s]


168 chunks stored


### **Step 3: Test the search**
A **lower score = closer**. Check that the right section comes first for each question.

In [4]:
def test_search(question, k=3):
    print("\nQ:", question)
    for doc, score in vectorstore.similarity_search_with_score(question, k=k):
        print(f"  s.{doc.metadata['section_no']:5} {doc.metadata['section_title'][:50]:50} score={score:.3f}")

test_search("What is the maximum normal working week?")
test_search("How much maternity leave do I get?")
test_search("How much notice must I give to resign?")
test_search("Can my boss make me work 12 hours a day?")


Q: What is the maximum normal working week?
  s.60    Work on rest day                                   score=0.626
  s.60A   Hours of work and working at night                 score=0.631
  s.60C   Shift work                                         score=0.634

Q: How much maternity leave do I get?
  s.37    Length of eligible period and entitlement to mater score=0.520
  s.37    Length of eligible period and entitlement to mater score=0.555
  s.37    Length of eligible period and entitlement to mater score=0.558

Q: How much notice must I give to resign?
  s.12    Notice of termination of contract                  score=0.532
  s.63A   Duty to give notice and other information          score=0.573
  s.13    Termination of contract without notice             score=0.622

Q: Can my boss make me work 12 hours a day?
  s.60A   Hours of work and working at night                 score=0.528
  s.60C   Shift work                                         score=0.576
  s.60A   Hours of work a

### **What to check**
- `maximum normal working week` should return **s.60A** near the top.
- `maternity leave` should return **s.37**.
- `notice` should return **s.12**.

If the right section is not in the top 3, try a different `k`, a different embedding model, or look at how that section was chunked in Project 1.